<a href="https://colab.research.google.com/github/cincasoler/IMAGEN/blob/main/IAHablante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai google-api-python-client gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


In [12]:
# Importamos todas las librerías necesarias
import google.generativeai as genai
from googleapiclient.discovery import build
import requests
from bs4 import BeautifulSoup
from gtts import gTTS
from IPython.display import Audio, display

# --- CONFIGURACIÓN DE LAS TRES LLAVES ---
# Reemplaza estas cadenas con tus credenciales reales
GEMINI_API_KEY = 'AQUÍ_VA_TU_API_KEY_DE_GEMINI'
SEARCH_API_KEY = 'AQUÍ_VA_TU_API_KEY_DE_GOOGLE_CLOUD'
SEARCH_ENGINE_ID = 'AQUÍ_VA_TU_ID_DE_MOTOR_DE_BÚSQUEDA_CX'

# Configurar el modelo de Gemini
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')


# --- FUNCIÓN 1: BUSCAR EN GOOGLE ---
def buscar_en_google(query, num_results=3):
    print(f"🔍 Buscando en Google: '{query}'")
    try:
        service = build("customsearch", "v1", developerKey=SEARCH_API_KEY)
        res = service.cse().list(q=query, cx=SEARCH_ENGINE_ID, num=num_results).execute()
        return [item['link'] for item in res.get('items', [])]
    except Exception as e:
        print(f"Error en la búsqueda: {e}")
        return []

# --- FUNCIÓN 2: EXTRAER TEXTO DE LAS PÁGINAS ---
def extraer_y_combinar_texto(urls):
    print(" G.I.R.A. (Getting Information Ready Assistant) Extrayendo información...")
    contexto_completo = ""
    for url in urls:
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            for script_or_style in soup(['script', 'style']):
                script_or_style.decompose()
            contexto_completo += soup.get_text(separator=' ', strip=True) + "\n\n"
        except Exception as e:
            print(f"No se pudo leer la URL {url}. Error: {e}")
    return contexto_completo

# --- FUNCIÓN 3: GENERAR RESPUESTA CON IA ---
def generar_respuesta_ia(contexto, pregunta):
    print("🧠 Cerebro de IA generando respuesta...")
    prompt = f"""
    Basado EXCLUSIVAMENTE en el siguiente contexto, responde la pregunta de forma clara y directa.
    Contexto:
    {contexto}
    ---
    Pregunta: {pregunta}
    """
    try:
        respuesta = model.generate_content(prompt)
        return respuesta.text
    except Exception as e:
        return f"Error al generar respuesta: {e}"

# --- FUNCIÓN 4: GENERAR Y REPRODUCIR AUDIO ---
def generar_y_reproducir_audio(texto, lang='es'):
    print("🗣️ Preparando la voz...")
    try:
        tts = gTTS(text=texto, lang=lang)
        tts.save("respuesta.mp3")
        display(Audio("respuesta.mp3", autoplay=True))
    except Exception as e:
        print(f"Error al generar audio: {e}")

# --- FUNCIÓN PRINCIPAL ---
def main():
    pregunta = input("Hola, soy GIRA. ¿Qué te gustaría saber hoy? ")
    if not pregunta:
        print("No has hecho ninguna pregunta.")
        return

    urls = buscar_en_google(pregunta)
    if not urls:
        error_msg = "No pude encontrar información relevante en internet sobre tu pregunta."
        print(error_msg)
        generar_y_reproducir_audio(error_msg)
        return

    contexto = extraer_y_combinar_texto(urls)
    if not contexto:
        error_msg = "Pude encontrar páginas, pero no logré extraer su contenido."
        print(error_msg)
        generar_y_reproducir_audio(error_msg)
        return

    respuesta_texto = generar_respuesta_ia(contexto, pregunta)

    print("\n--- Respuesta ---")
    print(respuesta_texto)
    print("-----------------\n")

    generar_y_reproducir_audio(respuesta_texto)

# --- EJECUTAR EL PROGRAMA ---
if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'gtts'